### Imports

In [ ]:
import glob
import pandas as pd
import os
import numpy as np

### Network layer features

In [3]:
# 9 selected network layer features
network_features = [
    "Fwd Header Len",
    "Init Bwd Win Byts",
    "Flow IAT Min",
    "Subflow Fwd Pkts",
    "Init Fwd Win Byts",
    "Tot Fwd Pkts",
    "Subflow Bwd Pkts",
    "Tot Bwd Pkts",
    "Bwd Header Len"
]

In [4]:
# 11 selected application layer features
application_features = [
    "Bwd Pkt Len Std",
    "Fwd Pkt Len Std",
    "Fwd Seg Size Avg",
    "Fwd Pkt Len Mean",
    "Fwd Pkt Len Max",
    "Subflow Fwd Byts",
    "TotLen Fwd Pkts",
    "Bwd Pkt Len Mean",
    "Bwd Seg Size Avg",
    "Pkt Len Mean",
    "TotLen Bwd Pkts"
]

### Uniform samapling approach

In [ ]:
data_path = './*.csv'
files = glob.glob(data_path)

sampled_dfs = []
sample_frac = 0.25

for file in files:
    print(f"Processing {os.path.basename(file)}...")
    for chunk in pd.read_csv(file, chunksize=200000, usecols=network_features + ['Label'], low_memory=False):
        sampled_chunk = chunk.sample(frac=sample_frac, random_state=42)
        sampled_dfs.append(sampled_chunk)
        del chunk

df25 = pd.concat(sampled_dfs, ignore_index=True)
print(df25.shape)
df25['Label'].value_counts(normalize=True)

Processing 02-14-2018.csv...
Processing 02-15-2018.csv...
Processing 02-16-2018.csv...
Processing 02-20-2018.csv...
Processing 02-21-2018.csv...
Processing 02-22-2018.csv...
Processing 02-23-2018.csv...
Processing 02-28-2018.csv...
Processing 03-01-2018.csv...
Processing 03-02-2018.csv...
(4058252, 10)


Label
Benign                      0.830718
DDOS attack-HOIC            0.042268
DDoS attacks-LOIC-HTTP      0.035480
DoS attacks-Hulk            0.028510
Bot                         0.017579
FTP-BruteForce              0.011924
SSH-Bruteforce              0.011540
Infilteration               0.009990
DoS attacks-SlowHTTPTest    0.008590
DoS attacks-GoldenEye       0.002528
DoS attacks-Slowloris       0.000703
DDOS attack-LOIC-UDP        0.000106
Brute Force -Web            0.000044
Brute Force -XSS            0.000013
SQL Injection               0.000004
Label                       0.000003
Name: proportion, dtype: float64

In [ ]:
data_path = './*.csv'
files = glob.glob(data_path)

sampled_dfs = []
sample_frac = 0.50

for file in files:
    print(f"Processing {os.path.basename(file)}...")
    for chunk in pd.read_csv(file, chunksize=200000, usecols=network_features + ['Label'], low_memory=False):
        sampled_chunk = chunk.sample(frac=sample_frac, random_state=42)
        sampled_dfs.append(sampled_chunk)
        del chunk

df50 = pd.concat(sampled_dfs, ignore_index=True)
print(df50.shape)
df50['Label'].value_counts(normalize=True)

Processing 02-14-2018.csv...
Processing 02-15-2018.csv...
Processing 02-16-2018.csv...
Processing 02-20-2018.csv...
Processing 02-21-2018.csv...
Processing 02-22-2018.csv...
Processing 02-23-2018.csv...
Processing 02-28-2018.csv...
Processing 03-01-2018.csv...
Processing 03-02-2018.csv...
(8116504, 10)


Label
Benign                      0.830741
DDOS attack-HOIC            0.042272
DDoS attacks-LOIC-HTTP      0.035489
DoS attacks-Hulk            0.028474
Bot                         0.017567
FTP-BruteForce              0.011913
SSH-Bruteforce              0.011544
Infilteration               0.010000
DoS attacks-SlowHTTPTest    0.008602
DoS attacks-GoldenEye       0.002530
DoS attacks-Slowloris       0.000692
DDOS attack-LOIC-UDP        0.000108
Brute Force -Web            0.000045
Brute Force -XSS            0.000014
SQL Injection               0.000006
Label                       0.000003
Name: proportion, dtype: float64

### Downsample the benign

=> Only extract the 5% of the benign from each chunks

In [7]:
data_path = "./*.csv"
files = glob.glob(data_path)

sampled_dfs = []

BENIGN_FRAC = 0.05     # keep only 5% of benign
ATTACK_FRAC = 1.0     # keep all attacks

for file in files:
    print(f"Processing {os.path.basename(file)}...")
    
    for chunk in pd.read_csv(
        file,
        chunksize=200_000,
        usecols=network_features + ["Label"],
        low_memory=False
    ):
        # Split by class
        benign = chunk[chunk["Label"] == "Benign"]
        attack = chunk[chunk["Label"] != "Benign"]

        # Downsample benign
        benign_sampled = benign.sample(
            frac=BENIGN_FRAC,
            random_state=42
        ) if len(benign) > 0 else benign

        # Keep all attacks
        sampled_chunk = pd.concat([benign_sampled, attack], ignore_index=True)
        sampled_dfs.append(sampled_chunk)

        del chunk

df_balanced = pd.concat(sampled_dfs, ignore_index=True)

print(df_balanced.shape)
print(df_balanced["Label"].value_counts(normalize=True))


Processing 02-14-2018.csv...
Processing 02-15-2018.csv...
Processing 02-16-2018.csv...
Processing 02-20-2018.csv...
Processing 02-21-2018.csv...
Processing 02-22-2018.csv...
Processing 02-23-2018.csv...
Processing 02-28-2018.csv...
Processing 03-01-2018.csv...
Processing 03-02-2018.csv...
(3422531, 10)
Label
DDOS attack-HOIC            0.200440
Benign                      0.197000
DDoS attacks-LOIC-HTTP      0.168352
DoS attacks-Hulk            0.134962
Bot                         0.083620
FTP-BruteForce              0.056496
SSH-Bruteforce              0.054810
Infilteration               0.047314
DoS attacks-SlowHTTPTest    0.040873
DoS attacks-GoldenEye       0.012128
DoS attacks-Slowloris       0.003211
DDOS attack-LOIC-UDP        0.000505
Brute Force -Web            0.000179
Brute Force -XSS            0.000067
SQL Injection               0.000025
Label                       0.000017
Name: proportion, dtype: float64


=> Take the same amout of benign and attack in each chunks

In [8]:
data_path = "./*.csv"
files = glob.glob(data_path)

sampled_dfs = []

for file in files:
    print(f"Processing {os.path.basename(file)}...")
    
    for chunk in pd.read_csv(
        file,
        chunksize=200_000,
        usecols=network_features + ["Label"],
        low_memory=False
    ):
        # Split by class
        benign = chunk[chunk["Label"] == "Benign"]
        attack = chunk[chunk["Label"] != "Benign"]

        MAX_BENIGN = len(attack)

        # Downsample benign
        benign_sampled = benign.sample(
            n=min(len(benign), MAX_BENIGN),
            random_state=42
        )

        # Keep all attacks
        sampled_chunk = pd.concat([benign_sampled, attack], ignore_index=True)
        sampled_dfs.append(sampled_chunk)

        del chunk

df_balanced = pd.concat(sampled_dfs, ignore_index=True)

print(df_balanced.shape)
print(df_balanced["Label"].value_counts(normalize=True))


Processing 02-14-2018.csv...
Processing 02-15-2018.csv...
Processing 02-16-2018.csv...
Processing 02-20-2018.csv...
Processing 02-21-2018.csv...
Processing 02-22-2018.csv...
Processing 02-23-2018.csv...
Processing 02-28-2018.csv...
Processing 03-01-2018.csv...
Processing 03-02-2018.csv...
(3915213, 10)
Label
Benign                      0.298047
DDOS attack-HOIC            0.175217
DDoS attacks-LOIC-HTTP      0.147167
DoS attacks-Hulk            0.117979
Bot                         0.073097
FTP-BruteForce              0.049387
SSH-Bruteforce              0.047913
Infilteration               0.041360
DoS attacks-SlowHTTPTest    0.035730
DoS attacks-GoldenEye       0.010602
DoS attacks-Slowloris       0.002807
DDOS attack-LOIC-UDP        0.000442
Brute Force -Web            0.000156
Brute Force -XSS            0.000059
SQL Injection               0.000022
Label                       0.000015
Name: proportion, dtype: float64


=> To get the 1:1 ratio between benign and attacks

### Network Feature Dataset

In [5]:
data_path = "./*.csv"
files = glob.glob(data_path)

DESIRED_BENIGN_RATIO = 0.6

# Step 1: Collect ALL attack rows and count them
attack_dfs = []
total_attacks = 0

print("First pass: Collecting all attack instances...")
for file in files:
    print(f"Scanning {os.path.basename(file)}...")
    for chunk in pd.read_csv(file, chunksize=200_000, usecols=network_features + ["Label"], low_memory=False):
        attack = chunk[chunk["Label"] != "Benign"] 
        attack = attack[attack["Label"] != "Label"]
        if len(attack) > 0:
            attack_dfs.append(attack)
        total_attacks += len(attack)
        del chunk

# Calculate how many benign rows we need globally
target_benign_count = int(total_attacks * DESIRED_BENIGN_RATIO / (1 - DESIRED_BENIGN_RATIO))

print(f"Total attacks: {total_attacks}")
print(f"Target benign count for {DESIRED_BENIGN_RATIO*100}% benign: {target_benign_count}")

# Step 2: Sample benign rows globally
benign_samples = []
benign_needed = target_benign_count

print("Second pass: Sampling benign rows globally...")
for file in files:
    for chunk in pd.read_csv(file, chunksize=200_000, usecols=network_features + ["Label"], low_memory=False):
        benign = chunk[chunk["Label"] == "Benign"]
        if len(benign) > 0 and benign_needed > 0:
            sample_n = min(len(benign), benign_needed)
            sampled = benign.sample(n=sample_n, random_state=42)
            benign_samples.append(sampled)
            benign_needed -= sample_n
        del chunk

    if benign_needed <= 0:
        break  # Stop early if we have enough benign

# Combine everything
print("Combining data...")
df_attacks = pd.concat(attack_dfs, ignore_index=True)
df_benign = pd.concat(benign_samples, ignore_index=True) if benign_samples else pd.DataFrame()

df_balanced = pd.concat([df_benign, df_attacks], ignore_index=True)

print(df_balanced.shape)
print("\nFinal distribution:")
print(df_balanced["Label"].value_counts(normalize=True).round(6))

First pass: Collecting all attack instances...
Scanning 02-14-2018.csv...
Scanning 02-15-2018.csv...
Scanning 02-16-2018.csv...
Scanning 02-20-2018.csv...
Scanning 02-21-2018.csv...
Scanning 02-22-2018.csv...
Scanning 02-23-2018.csv...
Scanning 02-28-2018.csv...
Scanning 03-01-2018.csv...
Scanning 03-02-2018.csv...
Total attacks: 2748235
Target benign count for 60.0% benign: 4122352
Second pass: Sampling benign rows globally...
Combining data...
(6870587, 10)

Final distribution:
Label
Benign                      0.600000
DDOS attack-HOIC            0.099848
DDoS attacks-LOIC-HTTP      0.083863
DoS attacks-Hulk            0.067230
Bot                         0.041655
FTP-BruteForce              0.028143
SSH-Bruteforce              0.027303
Infilteration               0.023569
DoS attacks-SlowHTTPTest    0.020361
DoS attacks-GoldenEye       0.006041
DoS attacks-Slowloris       0.001600
DDOS attack-LOIC-UDP        0.000252
Brute Force -Web            0.000089
Brute Force -XSS            

Major attacks (HOIC, LOIC-HTTP, Hulk, Bot) still dominate proportionally — realistic.

Medium ones (FTP/SSH BruteForce, Infiltration, SlowHTTPTest) well-represented.

Rare ones (GoldenEye, Slowloris, LOIC-UDP, Web attacks, SQL Injection) preserved with their original counts — critical for multi-class evaluation.

### Shuffle the data for better training result

In [6]:
# After creating df_balanced
print("Shuffling the dataset...")
df_balanced = df_balanced.sample(frac=1.0, random_state=42).reset_index(drop=True)

print("Shuffled distribution check (first few labels):")
print(df_balanced['Label'].head(20))

Shuffling the dataset...
Shuffled distribution check (first few labels):
0     DoS attacks-SlowHTTPTest
1                       Benign
2             DoS attacks-Hulk
3       DDoS attacks-LOIC-HTTP
4                       Benign
5       DDoS attacks-LOIC-HTTP
6                       Benign
7                       Benign
8                       Benign
9                       Benign
10            DoS attacks-Hulk
11                      Benign
12                      Benign
13                      Benign
14                      Benign
15                      Benign
16                      Benign
17              FTP-BruteForce
18            DoS attacks-Hulk
19                      Benign
Name: Label, dtype: object


In [7]:
# Save to CSV
df_balanced.to_csv("./newDataset/balanced_network_data.csv", index=False)

In [8]:
balanced = pd.read_csv("./newDataset/balanced_network_data.csv")
print("Records:", balanced.shape[0])

Records: 6870587


In [9]:
sqli = balanced[balanced['Label'] == 'SQL Injection']
print(f"sql injection records: {sqli.shape[0]}")

brute_xss = balanced[balanced['Label'] == 'Brute Force -XSS']
print(f"brute force XSS records: {brute_xss.shape[0]}")

brute_web = balanced[balanced['Label'] == 'Brute Force -Web']
print(f"brute force web records: {brute_web.shape[0]}")

loic = balanced[balanced['Label'] == 'DDOS attack-LOIC-UDP']
print(f"ddos records: {loic.shape[0]}")

slowloris = balanced[balanced['Label'] == 'DoS attacks-Slowloris']
print(f"slowloris records: {slowloris.shape[0]}")

goldeneye = balanced[balanced['Label'] == 'DoS attacks-GoldenEye']
print(f"goldeneye records: {goldeneye.shape[0]}")

sql injection records: 87
brute force XSS records: 230
brute force web records: 611
ddos records: 1730
slowloris records: 10990
goldeneye records: 41508


### Application Feature dataset

In [10]:
data_path = "./*.csv"
files = glob.glob(data_path)

for file in files:
    cols = pd.read_csv(file, nrows=0).columns.tolist()
    print(f"{os.path.basename(file)} columns: {cols}")

02-14-2018.csv columns: ['Dst Port', 'Protocol', 'Timestamp', 'Flow Duration', 'Tot Fwd Pkts', 'Tot Bwd Pkts', 'TotLen Fwd Pkts', 'TotLen Bwd Pkts', 'Fwd Pkt Len Max', 'Fwd Pkt Len Min', 'Fwd Pkt Len Mean', 'Fwd Pkt Len Std', 'Bwd Pkt Len Max', 'Bwd Pkt Len Min', 'Bwd Pkt Len Mean', 'Bwd Pkt Len Std', 'Flow Byts/s', 'Flow Pkts/s', 'Flow IAT Mean', 'Flow IAT Std', 'Flow IAT Max', 'Flow IAT Min', 'Fwd IAT Tot', 'Fwd IAT Mean', 'Fwd IAT Std', 'Fwd IAT Max', 'Fwd IAT Min', 'Bwd IAT Tot', 'Bwd IAT Mean', 'Bwd IAT Std', 'Bwd IAT Max', 'Bwd IAT Min', 'Fwd PSH Flags', 'Bwd PSH Flags', 'Fwd URG Flags', 'Bwd URG Flags', 'Fwd Header Len', 'Bwd Header Len', 'Fwd Pkts/s', 'Bwd Pkts/s', 'Pkt Len Min', 'Pkt Len Max', 'Pkt Len Mean', 'Pkt Len Std', 'Pkt Len Var', 'FIN Flag Cnt', 'SYN Flag Cnt', 'RST Flag Cnt', 'PSH Flag Cnt', 'ACK Flag Cnt', 'URG Flag Cnt', 'CWE Flag Count', 'ECE Flag Cnt', 'Down/Up Ratio', 'Pkt Size Avg', 'Fwd Seg Size Avg', 'Bwd Seg Size Avg', 'Fwd Byts/b Avg', 'Fwd Pkts/b Avg', 'Fw

In [11]:
data_path = "./*.csv"
files = glob.glob(data_path)

DESIRED_BENIGN_RATIO = 0.6

# Step 1: Collect ALL attack rows and count them
attack_dfs = []
total_attacks = 0

print("First pass: Collecting all attack instances...")
for file in files:
    print(f"Scanning {os.path.basename(file)}...")
    for chunk in pd.read_csv(file, chunksize=200_000, usecols=application_features + ["Label"], low_memory=False):
        attack = chunk[chunk["Label"] != "Benign"] 
        attack = attack[attack["Label"] != "Label"]
        if len(attack) > 0:
            attack_dfs.append(attack)
        total_attacks += len(attack)
        del chunk

# Calculate how many benign rows we need globally
target_benign_count = int(total_attacks * DESIRED_BENIGN_RATIO / (1 - DESIRED_BENIGN_RATIO))

print(f"Total attacks: {total_attacks}")
print(f"Target benign count for {DESIRED_BENIGN_RATIO*100}% benign: {target_benign_count}")

# Step 2: Sample benign rows globally
benign_samples = []
benign_needed = target_benign_count

print("Second pass: Sampling benign rows globally...")
for file in files:
    for chunk in pd.read_csv(file, chunksize=200_000, usecols=application_features + ["Label"], low_memory=False):
        benign = chunk[chunk["Label"] == "Benign"]
        if len(benign) > 0 and benign_needed > 0:
            sample_n = min(len(benign), benign_needed)
            sampled = benign.sample(n=sample_n, random_state=42)
            benign_samples.append(sampled)
            benign_needed -= sample_n
        del chunk

    if benign_needed <= 0:
        break  # Stop early if we have enough benign

# Combine everything
print("Combining data...")
df_attacks = pd.concat(attack_dfs, ignore_index=True)
df_benign = pd.concat(benign_samples, ignore_index=True) if benign_samples else pd.DataFrame()

df_balanced = pd.concat([df_benign, df_attacks], ignore_index=True)

print(df_balanced.shape)
print("\nFinal distribution:")
print(df_balanced["Label"].value_counts(normalize=True).round(6))

First pass: Collecting all attack instances...
Scanning 02-14-2018.csv...
Scanning 02-15-2018.csv...
Scanning 02-16-2018.csv...
Scanning 02-20-2018.csv...
Scanning 02-21-2018.csv...
Scanning 02-22-2018.csv...
Scanning 02-23-2018.csv...
Scanning 02-28-2018.csv...
Scanning 03-01-2018.csv...
Scanning 03-02-2018.csv...
Total attacks: 2748235
Target benign count for 60.0% benign: 4122352
Second pass: Sampling benign rows globally...
Combining data...
(6870587, 12)

Final distribution:
Label
Benign                      0.600000
DDOS attack-HOIC            0.099848
DDoS attacks-LOIC-HTTP      0.083863
DoS attacks-Hulk            0.067230
Bot                         0.041655
FTP-BruteForce              0.028143
SSH-Bruteforce              0.027303
Infilteration               0.023569
DoS attacks-SlowHTTPTest    0.020361
DoS attacks-GoldenEye       0.006041
DoS attacks-Slowloris       0.001600
DDOS attack-LOIC-UDP        0.000252
Brute Force -Web            0.000089
Brute Force -XSS            

In [12]:
# After creating df_balanced
print("Shuffling the dataset...")
df_balanced = df_balanced.sample(frac=1.0, random_state=42).reset_index(drop=True)

print("Shuffled distribution check (first few labels):")
print(df_balanced['Label'].head(20))

Shuffling the dataset...
Shuffled distribution check (first few labels):
0     DoS attacks-SlowHTTPTest
1                       Benign
2             DoS attacks-Hulk
3       DDoS attacks-LOIC-HTTP
4                       Benign
5       DDoS attacks-LOIC-HTTP
6                       Benign
7                       Benign
8                       Benign
9                       Benign
10            DoS attacks-Hulk
11                      Benign
12                      Benign
13                      Benign
14                      Benign
15                      Benign
16                      Benign
17              FTP-BruteForce
18            DoS attacks-Hulk
19                      Benign
Name: Label, dtype: object


In [13]:
df_balanced.to_csv("./newDataset/balanced_application_data.csv", index=False)

In [14]:
balanced_application = pd.read_csv("./newDataset/balanced_application_data.csv")
print("Records:", balanced_application.shape[0])

Records: 6870587
